# YouTube-bot Canonical Mobile Video Creator

**Use this notebook only.** It is self-contained, creates visible animated frames, Persian/English narration, subtitles, an MP4, and a ZIP under `/content`. It does not clone the repository, use stale commits, or call `gTTS(lang='fa')`.

Run the cells top to bottom. If online Persian TTS is unavailable, it automatically uses local `espeak-ng` or a clearly labelled silent-audio fallback.

In [ ]:
#@title 1) Settings
TOPIC = 'چرا آسمان آبی است؟' #@param {type:'string'}
LANGUAGE = 'fa' #@param ['fa', 'en']
EPISODE_ID = 'mobile-final-001' #@param {type:'string'}
VERTICAL = False #@param {type:'boolean'}
print({'topic': TOPIC, 'language': LANGUAGE, 'episode': EPISODE_ID, 'vertical': VERTICAL})

In [ ]:
#@title 2) Build, validate, and download
import asyncio, json, math, random, shutil, subprocess, sys, textwrap, wave, struct, zipfile
from pathlib import Path
from PIL import Image, ImageDraw, ImageFont
from google.colab import files
root = Path('/content') / EPISODE_ID
if root.exists(): shutil.rmtree(root)
frames = root / 'frames'; frames.mkdir(parents=True)
W, H = ((720, 1280) if VERTICAL else (1280, 720))
if LANGUAGE == 'fa':
    script = [f'امروز درباره {TOPIC} صحبت می کنیم.', 'پشت این موضوع یک داستان علمی جذاب وجود دارد.', 'با دیدن سازوکار آن، پاسخ کاملاً روشن می شود.', f'حالا می دانیم که {TOPIC} فقط یک واقعیت ساده نیست.', 'برای داستان های علمی بیشتر، کانال را دنبال کنید.']
else:
    script = [f'Today we explore {TOPIC}.', 'Behind this topic is a surprising scientific story.', 'Once we see the mechanism, the answer becomes clear.', f'Now we know that {TOPIC} is more than a simple fact.', 'Follow for more evidence-backed science stories.']
try:
    import arabic_reshaper
    from bidi.algorithm import get_display
    shape = lambda s: get_display(arabic_reshaper.reshape(s)) if LANGUAGE == 'fa' else s
except Exception:
    shape = lambda s: s
font_path = '/usr/share/fonts/truetype/noto/NotoNaskhArabic-Bold.ttf' if LANGUAGE == 'fa' else '/usr/share/fonts/truetype/dejavu/DejaVuSans-Bold.ttf'
if not Path(font_path).exists():
    font_path = '/usr/share/fonts/truetype/dejavu/DejaVuSans-Bold.ttf'
    if not Path(font_path).exists(): subprocess.run(['bash','-lc','apt-get -qq update && apt-get -qq install -y fonts-dejavu'], check=True)
font = ImageFont.truetype(font_path, max(30, int(W * 0.045)))
small = ImageFont.truetype(font_path, max(18, int(W * 0.018)))
duration = max(20.0, len(' '.join(script)) / 8.0); fps = 12; total = max(1, int(duration * fps)); random.seed(7)
particles = [(random.randrange(W), random.randrange(H), random.randrange(2, 8), random.random() * 6.28) for _ in range(50)]
for i in range(total):
    t = i / fps; scene = min(len(script)-1, int(t / (duration / len(script))))
    img = Image.new('RGB', (W, H), (7, 15, 45)); draw = ImageDraw.Draw(img)
    for y in range(H): draw.line((0, y, W, y), fill=(int(8+22*y/H), int(20+45*y/H), int(62+80*y/H)))
    for x, y, r, p in particles:
        xx = int((x + 45 * math.sin(t*.7+p)) % W); yy = int((y + 35 * math.cos(t*.5+p)) % H); draw.ellipse((xx-r, yy-r, xx+r, yy+r), fill=(70,170,240))
    lines = []
    for part in script[scene].split():
        candidate = (' '.join(lines[-1].split() + [part])) if lines else part
        if len(candidate) > (24 if not VERTICAL else 14) and lines: lines.append(part)
        elif lines: lines[-1] = candidate
        else: lines.append(part)
    y0 = int(H * .43)
    for line in lines:
        shown = shape(line); box = draw.textbbox((0,0), shown, font=font); x = (W-(box[2]-box[0]))//2; draw.text((x+3,y0+3), shown, font=font, fill='black'); draw.text((x,y0), shown, font=font, fill='white'); y0 += int(font.size*1.3)
    draw.text((int(W*.05), int(H*.91)), shape(f'SCENE {scene+1}/{len(script)}  •  YouTube-bot'), font=small, fill=(180,235,255))
    img.save(frames / f'frame-{i:05d}.jpg', quality=90)
audio = root / 'narration.mp3'; voice = 'none'
try:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'edge-tts'], check=True)
    import edge_tts
    voice = 'fa-IR-DilaraNeural' if LANGUAGE == 'fa' else 'en-US-AriaNeural'
    async def speak(): await edge_tts.Communicate(' '.join(script), voice).save(str(audio))
    asyncio.run(speak())
except Exception as err:
    print('Online TTS unavailable, trying local voice:', err)
    audio = root / 'narration.wav'
    local_voice = 'fa' if LANGUAGE == 'fa' else 'en'
    subprocess.run(['espeak-ng', '-v', local_voice, '-s', '145', '-w', str(audio), ' '.join(script)], check=False)
    if audio.exists() and audio.stat().st_size > 44: voice = f'espeak-ng {local_voice}'
if not audio.exists():
    audio = root / 'silence.wav'; rate = 22050; samples = int(rate * duration)
    with wave.open(str(audio), 'w') as wav: wav.setnchannels(1); wav.setsampwidth(2); wav.setframerate(rate); wav.writeframes(b'\x00\x00' * samples)
    voice = 'silent fallback — replace narration before publishing'
if shutil.which('ffmpeg') is None: subprocess.run(['bash','-lc','apt-get -qq update && apt-get -qq install -y ffmpeg'], check=True)
video = root / 'video.mp4'; command = ['ffmpeg','-y','-nostdin','-loglevel','error','-framerate',str(fps),'-i',str(frames/'frame-%05d.jpg'),'-i',str(audio),'-c:v','libx264','-c:a','aac','-shortest','-pix_fmt','yuv420p',str(video)]
run = subprocess.run(command, capture_output=True, text=True)
if run.returncode != 0 or not video.exists() or video.stat().st_size == 0: print(run.stderr); raise RuntimeError('MP4 render failed')
(root/'script.txt').write_text('\n\n'.join(script), encoding='utf-8'); (root/'captions.txt').write_text('\n'.join(script), encoding='utf-8'); (root/'manifest.json').write_text(json.dumps({'topic': TOPIC, 'language': LANGUAGE, 'voice': voice, 'animated': True, 'duration_seconds': duration, 'video': 'video.mp4'}, ensure_ascii=False, indent=2), encoding='utf-8')
archive = Path('/content') / f'{EPISODE_ID}.zip'
with zipfile.ZipFile(archive, 'w', zipfile.ZIP_DEFLATED) as z:
    for p in root.rglob('*'):
        if p.is_file(): z.write(p, p.relative_to(root.parent))
files.download(str(archive)); print(f'SUCCESS: {video} | {video.stat().st_size} bytes | voice={voice}')